# 01B — Seed Sensitivity: Patient-Level Splits

This notebook keeps the original 500-scan study cohort permanently frozen with
`COHORT_SEED = 42` and changes only `RUN_SEED`.

It **does not regenerate DICOM volumes, PyLIDC consensus masks, resized images,
or processed `.npz` files**. It only:

1. loads the frozen Seed-42 cohort,
2. creates a new patient-level 70/15/15 split,
3. verifies no patient leakage,
4. saves run-specific split CSVs, and
5. rebuilds the slice-level index using the existing processed `.npz` files.

Run first with `RUN_SEED = 1`, then change only that value to `2` and rerun.


In [11]:
# ============================================================
# SECTION 1 — Imports and experiment seeds
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

COHORT_SEED = 42      # PERMANENTLY FROZEN
RUN_SEED = 2          # Run 1; later change ONLY this to 2

print("Cohort seed:", COHORT_SEED)
print("Run seed:", RUN_SEED)


Cohort seed: 42
Run seed: 2


In [12]:
# ============================================================
# SECTION 2 — Project paths
# ============================================================

PROJECT_ROOT = Path(
    "/Volumes/Expansion/PLOS_2D_UNet_Revision"
)

PROCESSED_DIR = PROJECT_ROOT / "processed"
METADATA_DIR = PROJECT_ROOT / "metadata"
SPLITS_DIR = PROJECT_ROOT / "splits"

FROZEN_COHORT_PATH = (
    METADATA_DIR /
    f"selected_500_scan_cohort_seed{COHORT_SEED}.csv"
)

RUN_METADATA_DIR = (
    METADATA_DIR /
    f"seed{RUN_SEED}"
)

RUN_SPLITS_DIR = (
    SPLITS_DIR /
    f"seed{RUN_SEED}"
)

RUN_METADATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_SPLITS_DIR.mkdir(parents=True, exist_ok=True)

print("Frozen cohort:", FROZEN_COHORT_PATH)
print("Run metadata:", RUN_METADATA_DIR)
print("Run splits:", RUN_SPLITS_DIR)


Frozen cohort: /Volumes/Expansion/PLOS_2D_UNet_Revision/metadata/selected_500_scan_cohort_seed42.csv
Run metadata: /Volumes/Expansion/PLOS_2D_UNet_Revision/metadata/seed2
Run splits: /Volumes/Expansion/PLOS_2D_UNet_Revision/splits/seed2


In [13]:
# ============================================================
# SECTION 3 — Load and verify the frozen cohort
# ============================================================

if not FROZEN_COHORT_PATH.exists():
    raise FileNotFoundError(
        f"Frozen cohort not found: {FROZEN_COHORT_PATH}"
    )

cohort_df = pd.read_csv(FROZEN_COHORT_PATH)

required_columns = {
    "patient_id",
    "scan_id",
}

missing = required_columns - set(cohort_df.columns)

if missing:
    raise ValueError(
        f"Frozen cohort is missing columns: {sorted(missing)}"
    )

print("Frozen cohort scans:", len(cohort_df))
print("Frozen unique patients:", cohort_df["patient_id"].nunique())

assert len(cohort_df) == 500

cohort_df.head()


Frozen cohort scans: 500
Frozen unique patients: 498


,patient_id,scan_id,n_nodule_clusters,n_annotations,has_nodule_annotations,status,error_message
0,LIDC-IDRI-0001,12,1,4,True,OK,NaN
1,LIDC-IDRI-0003,14,4,13,True,OK,NaN
2,LIDC-IDRI-0004,15,1,4,True,OK,NaN
3,LIDC-IDRI-0006,17,4,8,True,OK,NaN
4,LIDC-IDRI-0007,18,2,5,True,OK,NaN


In [14]:
# ============================================================
# SECTION 4 — Create patient-level 70/15/15 split using RUN_SEED
# ============================================================

rng = np.random.default_rng(RUN_SEED)

unique_patients = np.array(
    sorted(cohort_df["patient_id"].unique())
)

rng.shuffle(unique_patients)

n_patients = len(unique_patients)

n_train = int(round(n_patients * 0.70))
n_val = int(round(n_patients * 0.15))

train_patients = unique_patients[:n_train]
val_patients = unique_patients[n_train:n_train + n_val]
test_patients = unique_patients[n_train + n_val:]

assert len(set(train_patients) & set(val_patients)) == 0
assert len(set(train_patients) & set(test_patients)) == 0
assert len(set(val_patients) & set(test_patients)) == 0

split_map = {
    **{pid: "train" for pid in train_patients},
    **{pid: "validation" for pid in val_patients},
    **{pid: "test" for pid in test_patients},
}

run_cohort_df = cohort_df.copy()
run_cohort_df["split"] = (
    run_cohort_df["patient_id"].map(split_map)
)

assert run_cohort_df["split"].notna().all()

print("\n=== PATIENT-LEVEL SPLIT ===")
print(run_cohort_df.groupby("split")["patient_id"].nunique())

print("\nScans per split:")
print(run_cohort_df["split"].value_counts())



=== PATIENT-LEVEL SPLIT ===
split
test           74
train         349
validation     75
Name: patient_id, dtype: int64

Scans per split:
split
train         351
validation     75
test           74
Name: count, dtype: int64


In [15]:
# ============================================================
# SECTION 5 — Explicit patient-leakage audit
# ============================================================

patient_split_counts = (
    run_cohort_df
    .groupby("patient_id")["split"]
    .nunique()
)

leaking_patients = patient_split_counts[
    patient_split_counts > 1
]

print(
    "Patients appearing in multiple splits:",
    len(leaking_patients)
)

assert len(leaking_patients) == 0

print(
    f"Seed {RUN_SEED} patient-level leakage check: PASSED"
)


Patients appearing in multiple splits: 0
Seed 2 patient-level leakage check: PASSED


In [16]:
# ============================================================
# SECTION 6 — Save run-specific patient split files
# ============================================================

preferred_split_columns = [
    "patient_id",
    "scan_id",
    "n_nodule_clusters",
    "n_annotations",
    "split",
]

split_columns = [
    column
    for column in preferred_split_columns
    if column in run_cohort_df.columns
]

cohort_split_df = (
    run_cohort_df[split_columns]
    .sort_values(["split", "patient_id", "scan_id"])
    .reset_index(drop=True)
)

patient_split_path = (
    RUN_SPLITS_DIR /
    f"patient_level_split_seed{RUN_SEED}.csv"
)

cohort_split_df.to_csv(
    patient_split_path,
    index=False,
)

for split_name in ["train", "validation", "test"]:
    split_subset = cohort_split_df[
        cohort_split_df["split"] == split_name
    ]

    split_subset.to_csv(
        RUN_SPLITS_DIR /
        f"{split_name}_scans_seed{RUN_SEED}.csv",
        index=False,
    )

print("Saved:", patient_split_path)


Saved: /Volumes/Expansion/PLOS_2D_UNet_Revision/splits/seed2/patient_level_split_seed2.csv


In [17]:
# ============================================================
# SECTION 7 — Verify all frozen-cohort processed files exist
# ============================================================

missing_npz = []

for _, row in run_cohort_df.iterrows():

    patient_id = row["patient_id"]
    scan_id = int(row["scan_id"])

    npz_path = (
        PROCESSED_DIR /
        f"{patient_id}_scan{scan_id}.npz"
    )

    if not npz_path.exists():
        missing_npz.append(str(npz_path))

print("Missing processed files:", len(missing_npz))

if missing_npz:
    print("First missing files:")
    for path in missing_npz[:10]:
        print(path)

assert len(missing_npz) == 0

print("Processed-data reuse check: PASSED")


Missing processed files: 0
Processed-data reuse check: PASSED


In [18]:
# ============================================================
# SECTION 8 — Rebuild slice-level index using existing .npz files
# ============================================================

slice_rows = []

for i, row in run_cohort_df.iterrows():

    patient_id = row["patient_id"]
    scan_id = int(row["scan_id"])
    split = row["split"]

    npz_path = (
        PROCESSED_DIR /
        f"{patient_id}_scan{scan_id}.npz"
    )

    with np.load(npz_path) as data:

        masks = data["consensus_mask"]

        if masks.ndim != 3:
            raise ValueError(
                f"{npz_path.name}: expected 3D mask, "
                f"got {masks.shape}"
            )

        n_slices = masks.shape[2]

        mask_pixels_per_slice = masks.sum(
            axis=(0, 1)
        )

        for z in range(n_slices):

            mask_pixels = int(
                mask_pixels_per_slice[z]
            )

            slice_rows.append({
                "patient_id": patient_id,
                "scan_id": scan_id,
                "split": split,
                "npz_path": str(npz_path),
                "slice_index": z,
                "is_positive": mask_pixels > 0,
                "mask_pixels": mask_pixels,
            })

    if (i + 1) % 50 == 0:
        print(
            f"Indexed {i + 1}/"
            f"{len(run_cohort_df)} scans..."
        )

slice_index_df = pd.DataFrame(slice_rows)

slice_index_path = (
    RUN_METADATA_DIR /
    f"slice_level_index_seed{RUN_SEED}.csv"
)

slice_index_df.to_csv(
    slice_index_path,
    index=False,
)

print("\nSlice-level index saved to:")
print(slice_index_path)

print("\nTotal slices:", len(slice_index_df))
print(
    "Positive slices:",
    int(slice_index_df["is_positive"].sum())
)

print("\nSlices by split:")
print(
    slice_index_df["split"].value_counts()
)

print("\nPositive slices by split:")
print(
    slice_index_df[
        slice_index_df["is_positive"]
    ]["split"].value_counts()
)


Indexed 50/500 scans...
Indexed 100/500 scans...
Indexed 150/500 scans...
Indexed 200/500 scans...
Indexed 250/500 scans...
Indexed 300/500 scans...
Indexed 350/500 scans...
Indexed 400/500 scans...
Indexed 450/500 scans...
Indexed 500/500 scans...

Slice-level index saved to:
/Volumes/Expansion/PLOS_2D_UNet_Revision/metadata/seed2/slice_level_index_seed2.csv

Total slices: 119816
Positive slices: 7716

Slices by split:
split
train         84825
validation    17716
test          17275
Name: count, dtype: int64

Positive slices by split:
split
train         5480
validation    1271
test           965
Name: count, dtype: int64


In [19]:
# ============================================================
# SECTION 9 — Final run audit
# ============================================================

patient_split_counts = (
    slice_index_df
    .groupby("patient_id")["split"]
    .nunique()
)

assert (patient_split_counts == 1).all()

assert set(
    slice_index_df["patient_id"].unique()
) == set(
    run_cohort_df["patient_id"].unique()
)

print("Cohort seed:", COHORT_SEED)
print("Run seed:", RUN_SEED)
print("Cohort scans:", len(run_cohort_df))
print(
    "Unique patients:",
    run_cohort_df["patient_id"].nunique()
)
print("Slice rows:", len(slice_index_df))
print("Leakage check: PASSED")
print("\nREADY FOR 02B_Seed_Sensitivity_Training.ipynb")


Cohort seed: 42
Run seed: 2
Cohort scans: 500
Unique patients: 498
Slice rows: 119816
Leakage check: PASSED

READY FOR 02B_Seed_Sensitivity_Training.ipynb
